# A Review of a Hierarchy of Box Models

**Python Translation of Mathematica Notebook:** `Work.nb`  
**Author:** Halldór Björnsson

---

## Abstract & Overview
This notebook presents a comprehensive analysis across a hierarchy of ocean climate box models of increasing physical sophistication:
1. **The Stommel (1961) Two-Box Model:** Analytical derivation of the advective feedback, Jacobian matrix, eigenvalue stability criteria, and the saddle-node tipping point.
2. **Rahmstorf's (1996) Four-Box Model — Version I (Interhemispheric):** Flow driven by the interhemispheric density gradient ($q = k (\rho_2 - \rho_1)$), full 4x4 Jacobian eigenvalue spectrum, and 3D stability surface.
3. **Rahmstorf's (1996) Four-Box Model — Version II (Intra-Basin):** Flow driven by intra-basin thermocline-to-polar density gradient ($q = k (\rho_2 - \rho_3)$), demonstrating how Southern Ocean freshwater forcing alters the stability of the upper branch.
4. **The Gnanadesikan (1999) Pycnocline Model:** A coupled thermohaline-wind model incorporating Southern Ocean Ekman transport and mesoscale eddy fluxes. Includes analytical implicit differentiation of the pycnocline cubic equation and a complete parameter sensitivity suite.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# Import the model classes
from stommel_model import StommelModel
from rahmstorf_4box import Rahmstorf4BoxModel
from gnanadesikan_model import GnanadesikanBoxModel


---
## 1. The Stommel (1961) Two-Box Model

### Mathematical Formulation
Two well-mixed ocean boxes (Equatorial warm/saline box 1, Polar cold/fresh box 2) are connected by an exchange flow $q$:
$$\dot{S}_1 = \frac{1}{V_1} [P_1 + q(S_2 - S_1)]$$
$$\dot{S}_2 = \frac{1}{V_2} [P_2 + q(S_1 - S_2)]$$
where $P_1 = -S_0 F$, $P_2 = S_0 F$, and $q = k [\beta(S_2 - S_1) - \alpha \Delta T]$.

At steady state:
$$\Delta S = S_2 - S_1 = -\frac{S_0 F}{q} \implies q^2 - \sigma q + \gamma F = 0$$
where $\sigma = k \alpha \Delta T$ and $\gamma = k \beta S_0$.

### Symbolic Jacobian & Eigenvalues
Under equal volume assumption ($V_1 = V_2 = V$), the Jacobian evaluated at steady state is:
$$M = \frac{1}{V} \begin{pmatrix} -(q + \gamma F / q) & \gamma F / q \\ \gamma F / q & -(q + \gamma F / q) \end{pmatrix}$$
giving eigenvalues:
$$\lambda_1 = -\frac{q}{V} \quad (\text{tracer decay}), \qquad \lambda_2 = -\frac{q^2 - \gamma F}{q V} \quad (\text{dynamical eigenvalue})$$

- Upper branch ($q > q_c = \sqrt{\gamma F}$): $\lambda_2 < 0 \implies$ **Linearly Stable**.
- Lower branch ($q < q_c$): $\lambda_2 > 0 \implies$ **Linearly Unstable** (Advective Instability).


In [ ]:
stommel = StommelModel(Delta_T=20.0, k=1.0)
print(f"Sigma (thermal driving):     {stommel.sigma:.2f} Sv")
print(f"Bifurcation threshold Fc:    {stommel.critical_flux:.4f} Sv")
print(f"Critical overturning qc:     {stommel.critical_flow:.2f} Sv")

# Plot stability curve and eigenvalue lambda_2
fig = stommel.plot_stability_curve()
plt.show()


---
## 2. Rahmstorf's (1996) Four-Box Model — Version I (Interhemispheric)

In Version I, flow is driven by the **interhemispheric density difference** between North Atlantic deep water sinking (Box 2) and South Atlantic surface waters (Box 1):
$$q = k (\rho_2 - \rho_1) = k [\beta(S_2 - S_1) - \alpha \Theta]$$

The 4 salinity ODEs are:
$$V_1 \dot{S}_1 = -S_0 F_1 + q(S_4 - S_1)$$
$$V_2 \dot{S}_2 = -S_0 F_2 + q(S_3 - S_2)$$
$$V_3 \dot{S}_3 = S_0 (F_1 + F_2) + q(S_1 - S_3)$$
$$V_4 \dot{S}_4 = q(S_2 - S_4)$$

### 4x4 Jacobian & Stability Surface
The 4x4 Jacobian matrix evaluated at steady state yields 4 eigenvalues:
- $\lambda_1 = 0$ (reflecting total salt conservation $\sum V_i S_i = \text{const}$)
- $\lambda_2 = -2q / V$
- $\lambda_{3, 4} = \frac{1}{2q V} \left[ -(2q^2 + \Gamma F_1 - \Gamma F_2) \pm \sqrt{(2q^2 + \Gamma F_1 - \Gamma F_2)^2 - 4q^2(2q^2 + 2\Gamma F_1)} \right]$


In [ ]:
r1 = Rahmstorf4BoxModel(version=1, Gamma=420.0, Sigma=18.0)
fig = r1.plot_stability_surface()
plt.show()


---
## 3. Rahmstorf's (1996) Four-Box Model — Version II (Intra-Basin)

In Version II, the flow rule is modified to depend on the **intra-basin density gradient** between the subpolar deep sinking box (Box 2) and the subtropical thermocline box (Box 3):
$$q = k (\rho_2 - \rho_3) = k [\beta(S_2 - S_3) - \alpha \Theta]$$

On the equilibrium surface:
$$q^2 + \Sigma q + \Gamma F_2 = 0$$
Notice the duality: the equilibrium curve now depends on $F_2$ (Southern Ocean freshwater flux), and the stability boundary depends on $F_1$!


In [ ]:
r2 = Rahmstorf4BoxModel(version=2, Gamma=420.0, Sigma=18.0)
fig = r2.plot_stability_surface()
plt.show()


---
## 4. The Gnanadesikan (1999) Wind-Driven Pycnocline Model

Anand Gnanadesikan (1999) connected the overturning circulation to Southern Ocean wind stress over Drake Passage:
$$q_n = q_{Ek} - q_{ed} + q_u = \frac{\tau L_x}{\rho_0 f_s} - \epsilon D + \frac{K_v A}{D}$$

### The AnandCubic:
$$\eta \Delta\rho D^3 + \epsilon D^2 - q_\tau D - \mu = 0$$

### Implicit Differentiation of the Pycnocline:
To form the Jacobian matrix without discarding pycnocline variations, `Work.nb` differentiates `AnandCubic` implicitly:
$$\frac{\partial D}{\partial S_2} = -\frac{\beta \eta D^3}{-q_\tau + 2 \epsilon D + 3 \eta \Delta\rho D^2}, \qquad \frac{\partial D}{\partial S_3} = -\frac{\partial D}{\partial S_2}$$


In [ ]:
gnan = GnanadesikanBoxModel()
fig = gnan.plot_sensitivity_suite()
plt.show()


---
## 5. Summary & Key Takeaways

1. **Hierarchy of Instability:**
   - In Stommel (1961), the stability boundary is cleanly determined by $q_c = \sigma / 2$.
   - In Rahmstorf (1996), introducing 4 boxes and interhemispheric/intra-basin flow laws reveals that southern hemisphere freshwater forcing ($F_2$) can induce instability even on the upper branch!
2. **Southern Ocean Control:**
   - Incorporating Gnanadesikan's pycnocline model transforms the symmetric Stommel parabola into an asymmetric, steep curve whose tipping threshold is dictated by wind stress and eddy mixing in the Southern Ocean.
